In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 通过load_dotenv()将.env中的变量加载为环境变量
# override=True表示：无论你当前的操作系统、终端或者虚拟环境中是否已经存在同名的环境变量，
# 都会强行用 .env 文件里写的值去覆盖它
load_dotenv(override=True)


# 从环境变量读取配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


# 创建LLM
model = init_chat_model(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    model="deepseek:deepseek-flash" # 冒号前面直定模型厂商，冒号后面指定模型
)

内置PII的使用

In [2]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email",strategy="redact",apply_to_input=True),
        PIIMiddleware("credit_card",strategy="mask",apply_to_input=True),
        PIIMiddleware("url",strategy="hash",apply_to_input=True),
        PIIMiddleware("mac_address",strategy="mask",apply_to_input=True),
        PIIMiddleware("ip",strategy="block",apply_to_input=True),
    ]
)


response = agent.invoke({
    "messages" : [HumanMessage("""
    帮我向 156168188@qq.com 发送一封邮件
    同时查看银行卡号： 5105-1051-0510-5100 的余额
    访问 https://localhost:12345
    确认这是不是 MAC地址： 11-11-11-11-11-11
    """)]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================


    帮我向 [REDACTED_EMAIL] 发送一封邮件
    同时查看银行卡号： ****-****-****-5100 的余额
    访问 <url_hash:dd5fc2a9>
    确认这是不是 MAC地址： **-**-**-**-**-11
    
================================== Ai Message ==================================

我无法直接替你完成这些操作：当前对话没有发送邮件、访问外部链接或查询银行卡余额的权限，也不应处理或核实敏感金融信息。

- **发送邮件**：请提供收件人、主题、正文和附件说明；我可以帮你起草/润色，你再在邮箱中发送。
- **银行卡余额**：无法查询。请通过银行官方 App、官网或客服查询；不要向他人提供完整卡号、验证码或密码。
- **访问链接**：我不能打开未知/哈希化链接。请提供完整 URL，并确认来源可信；谨防钓鱼链接。
- **MAC 地址确认**：`**-**-**-**-**-11` 是掩码形式，无法确认。完整 MAC 地址应为 6 组两位十六进制，如 `00-1A-2B-3C-4D-11` 或 `00:1A:2B:3C:4D:11`。末尾 `11` 合法，但其余部分被隐藏，不能判断是否为真实 MAC 地址。可在设备设置或命令行中用 `ipconfig /all`（Windows）、`ifconfig`/`ip link`（macOS/Linux）查看。


In [ ]:
try:
    agent.invoke({
        "messages": [HumanMessage("看看这个 IP 能不能 ping 通：192.168.10.1")]
    })
except Exception as e:
    print(f"检测到ip，抛出异常{e}")

自定义PII

In [4]:
import re
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage


# 自定义检测函数
def detect_phone_number(content: str):
    return [
        {
            "text": m.group(0), # 提取出具体匹配到的 11 位数字文本（例如 "13800138000"）
            "start": m.start(), # 这段数字在原文本中的“起始索引位置”（从 0 开始算）
            "end": m.end() # 这段数字在原文本中的“结束索引位置”
        } for m in re.finditer(r"[0-9]{11}", content)
    ]


agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("api_key", strategy="hash", apply_to_input=True, detector=r"sk-[a-zA-Z0-9]+"),
        PIIMiddleware("phone_number", strategy="mask", apply_to_input=True, detector=detect_phone_number)
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("""
    这是不是有效的 API_KEY： sk-awef23AFEfaafaefa
    帮我给这个号码打电话： 12345612345
    访问 https://localhost:12345
    """)]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================


    这是不是有效的 API_KEY： <api_key_hash:6c678cc0>
    帮我给这个号码打电话： ****2345
    访问 https://localhost:12345
    
================================== Ai Message ==================================

不能仅凭 `<api_key_hash:6c678cc0>` 判断 API_KEY 是否有效：它看起来是哈希/占位符，不是可验证的 API Key。需要到对应服务的验证接口或后台确认；不要公开真实 API Key。

也无法帮你给 `****2345` 打电话：号码被掩码且我没有拨号能力。  
`https://localhost:12345` 是**本机地址**，只能在运行该服务的本机/同一环境访问；从外部无法访问。请确认：
- 本机服务是否已启动并监听 12345 端口
- 是否应使用 `http://localhost:12345`
- HTTPS 证书是否自签名/可信

如果你把服务名称、API Key 的验证方式（不要发真实密钥）和完整本机报错贴出来，我可以帮你判断下一步。
